# Churn Risk Detection from Chats

### Conversational Banking Customer Service — Banking-AI

This notebook follows the Energy-AI philosophy: Beginner-friendly, synthetic data, EDA, modeling, and interpretation.

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of conversational banking and customer service terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the conversational banking and customer service problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Intent classification, sentiment analysis, chatbot retrieval, and customer service optimisation in banking.

## Step 1 — Banking Problem Introduction

Customers often express dissatisfaction in chats before closing their accounts. This notebook uses chat-derived features to predict churn risk.

**Primary stakeholders:** chatbot product owners, contact centre managers, UX researchers, and customer service leads.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

Synthetic data for 200 customers.

In [ ]:
n = 200

# Features: num_chats, avg_sentiment, num_complaints, last_wait_time
chats = RNG.integers(1, 20, n)
sentiment = RNG.uniform(-1, 1, n)
complaints = RNG.integers(0, 5, n)
wait_time = RNG.uniform(1, 30, n)
# Target: 1 if churn, 0 otherwise. High complaints and low sentiment increase churn risk.
churn = (complaints * 0.3 - sentiment * 0.5 + wait_time * 0.02 + RNG.normal(0, 0.1, n) > 0.5).astype(int)

df = pd.DataFrame({'num_chats': chats, 'sentiment': sentiment, 'complaints': complaints, 'wait_time': wait_time, 'churn': churn})

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Correlation with churn.

In [ ]:
print(df.groupby('churn').mean())

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = RandomForestClassifier()
model.fit(X_train, y_train)

In [ ]:
df['risk_prob'] = model.predict_proba(X)[:, 1]
high_risk = df[df['risk_prob'] > 0.7]
print(f'Number of high risk customers identified: {len(high_risk)}')

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

Identify high-risk customers.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end conversational banking and customer service workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Customers must be clearly informed when they are interacting with an AI system.
- Intent classification errors can misdirect customers, causing frustration and service failures.
- Conversational AI must escalate sensitive financial queries to human agents.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in conversational banking and customer service?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.